RQ2 -----------------------------------------------

Random Forest

General protective behaviour covid model

Dataset with state and covid 7 day rolling cases and deaths are considered here for the analysis.

In [2]:
# import libraries
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib


state_train_general_behav = pd.read_csv("state_train_general_behav.csv")
state_test_general_behav = pd.read_csv("state_test_general_behav.csv")


state_train_general_behav.columns

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'state',
       'household_size', 'Wellbeing', 'Perceived Severity',
       'Perceived Susceptibility', 'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment

In [3]:
states = sorted(state_train_general_behav["state"].unique()) # to get 8 states

# remove the target variables and identifiers
drop_cols = ['RecordNo', 'Date', 'state', 'face_mask_scale', 'face_mask_binary','general_protective_behavior_scale',
                'general_protective_behavior_binary','mandate_start_date']

cv = StratifiedKFold( # 5 fold cross validation
        n_splits=5,
        shuffle=True,
        random_state=42
)

  
cv_rf = Pipeline([   # standard scaler is not important in RF as RF makes decisions on the order of them not the scale
            ('ros', RandomOverSampler(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42,n_jobs=-1))
    ])   


params = { # apply the parameters to the rf step of the pipeline - rf__n_estimators  double underscore
        
            'rf__n_estimators': [250], 
            'rf__max_depth': [5,7],
            'rf__min_samples_split': [2,10],  # 2,10
            'rf__min_samples_leaf': [1,5],    # 1,5
            'rf__max_features': ['sqrt','log2']  # important for RF
    }

state_results = []

parameter_results = []

for state in states: # 8 RF state models are created trained 

    train_state = state_train_general_behav[state_train_general_behav["state"] == state]

    test_state = state_test_general_behav[state_test_general_behav["state"] == state]

    print(f"\nProcessing {state}")
    print(f"Training: {len(train_state)}")
    print(f"Testing: {len(test_state)}")


    # predictors
    x_train = train_state.drop(columns=drop_cols)
    x_test = test_state.drop(columns=drop_cols)

    x_train = x_train.astype(float)  # converting boolean to float 
    x_test = x_test.astype(float)

    print(x_train.columns)
    print(x_train.shape)

    # target variable 

    y_train = train_state["general_protective_behavior_binary"]
    y_test = test_state["general_protective_behavior_binary"]


    # RF model ---------------------------------------------------------


    grid = GridSearchCV( # tunes parameters
            cv_rf,
            params,
            cv=cv,
            scoring={  # evaluates all metrics
            'roc_auc': 'roc_auc',
            'accuracy': 'accuracy',
            'f1': 'f1'},
            refit='roc_auc',  # choose the best model
            return_train_score=False,


            n_jobs=-1 # use all available CPU cores for parallel processing
        )

    grid.fit(x_train, y_train)

    best_rf = grid.best_estimator_   # best model

    best_parameters = grid.best_params_   # best hyper parameters

    joblib.dump(best_parameters, f"RQ2_general_behav_{state}_RF_bestParameters_rolling.pkl")

    parameter_results.append({"State": state, **best_parameters})



    results_rf = pd.DataFrame(grid.cv_results_)

    pred = best_rf.predict(x_test)
    prob = best_rf.predict_proba(x_test)[:,1]


    accuracy = round(accuracy_score(y_test, pred),4)
    roc_auc = round(roc_auc_score(y_test, prob),4)
    f1 = round(f1_score(y_test, pred),4)


    print("Accuracy:", accuracy)
    print("ROC AUC:", roc_auc)
    print("F1:", f1)


    state_results.append({
        "State": state,
        "Accuracy": accuracy,
        "ROC AUC": roc_auc,
        "F1": f1
    })

    joblib.dump(best_rf, f"RQ2_general_behav_{state}_RF_rolling.pkl") # best model 
    results_rf.to_csv(f"RQ2_general_behav_{state}_RF_rolling_results.csv", index=False)



state_results = pd.DataFrame(state_results)
print(state_results)
state_results.to_csv("RQ2_general_behav_RF_state_results.csv", index=False)



parameter_results = pd.DataFrame(parameter_results)
parameter_results.to_csv("RQ2_general_behav_RF_best_parameters.csv", index=False)


Processing Australian Capital Territory
Training: 497
Testing: 131
Index(['Non-household contacts', 'age', 'household_size', 'Wellbeing',
       'Perceived Severity', 'Perceived Susceptibility',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_period', 'Isolate if unwell_Not sure',
       'Isolate if unwell_Yes', 'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment's response_Don't know',
       'Confidence in Goverment's response_No confidence at all',
       'Confidence in Goverment's 